# Caracterización y Análisis de Rendimiento PVT y Monte Carlo del Comparador Dinámico
---
**Autor:** Antigravity AI Partner  
**PDK:** GF180MCU 3.3V  
**Herramientas:** ngspice + Python (Jupyter)

Este cuaderno de Jupyter implementa un flujo automatizado para simular y caracterizar el comparador dinámico bajo:
1. **Esquinas PVT (Process, Voltage, Temperature):** Barrido combinatorio completo de 5 corners de transistores, 3 niveles de voltaje y 3 temperaturas (45 combinaciones).
2. **Simulación de Monte Carlo (200 Muestras):** Simulación estadística con variación local (mismatch de transistores) para el cálculo del rendimiento (*Yield*).

### Métricas Clave a Medir:
- **Retardo de propagación ($t_{pd}$):** Tiempo desde el flanco de bajada de la señal de reloj (`CLK_NEG`) hasta que la salida `On` cruza `VDD/2`.
- **Consumo de potencia promedio ($P_{avg}$):** Consumo promedio de la fuente de alimentación `VDD` durante la conmutación.
- **Offset dinámico ($V_{OS}$):** Medido usando el **Método de Rampa Lenta (Slow Ramp Method)** en el dominio del tiempo.

## 1. Configuración de Especificaciones y Parámetros Editables
Define aquí los límites de diseño (Specs) y la configuración de las simulaciones.

In [1]:
# =====================================================================
# ESPECIFICACIONES DE DISEÑO (LÍMITES MÁXIMOS)
# =====================================================================
SPEC_MAX_DELAY = 1.0e-9   # Retardo máximo de propagación (1.0 ns)
SPEC_MAX_POWER = 120e-6   # Consumo de potencia promedio máximo (120 uW)
SPEC_MAX_OFFSET = 15.0e-3 # Offset dinámico máximo (15.0 mV)

# =====================================================================
# PARÁMETROS DE SIMULACIÓN
# =====================================================================
SIMULATE_PVT = True      # True: Simula las esquinas PVT. False: Omite PVT para ahorrar tiempo.
MC_SAMPLES = 100         # Número de muestras para Monte Carlo
CLK_FREQ = 100e6          # Frecuencia de reloj (Hz) - ej. 500 MHz para alta resolución
CLK_PERIOD = 1 / CLK_FREQ  # Periodo de reloj (s) derivado automáticamente

# Rutas del PDK y del proyecto
PDK_PATH = '/foss/pdks/gf180mcuD'
NETLIST_DIR = '/foss/designs'
SIM_DIR = '/headless/.xschem/simulations'
import os
os.makedirs(SIM_DIR, exist_ok=True)

print("Especificaciones y parámetros configurados correctamente.")
print(f"Specs: Delay < {SPEC_MAX_DELAY*1e9:.1f} ns, Power < {SPEC_MAX_POWER*1e6:.1f} uW, Offset < {SPEC_MAX_OFFSET*1e3:.1f} mV")
print(f"¿Simular esquinas PVT?: {'SÍ' if SIMULATE_PVT else 'NO'}")
print(f"Muestras de Monte Carlo a simular: {MC_SAMPLES}")

Especificaciones y parámetros configurados correctamente.
Specs: Delay < 1.0 ns, Power < 120.0 uW, Offset < 15.0 mV
¿Simular esquinas PVT?: SÍ
Muestras de Monte Carlo a simular: 100


## Lector de Archivos Raw de Ngspice
Clase auxiliar para cargar los datos de simulación transitoria.

In [2]:
class NgspiceRawReader:
    def __init__(self, filepath):
        self.filepath = filepath
        self.variables = []
        self.data = {}
        self.load()

    def load(self):
        if not os.path.exists(self.filepath):
            raise FileNotFoundError(f"No se encontró el archivo: {self.filepath}")
            
        with open(self.filepath, 'r') as f:
            lines = f.readlines()

        # Buscar variables
        data_start = 0
        for idx, line in enumerate(lines):
            if line.startswith('Variables:'):
                j = idx + 1
                while not lines[j].startswith('Values:'):
                    parts = lines[j].strip().split()
                    if len(parts) >= 3:
                        self.variables.append(parts[1])
                    j += 1
            if line.startswith('Values:'):
                data_start = j + 1
                break

        # Inicializar contenedores de datos
        for var in self.variables:
            self.data[var] = []

        # Leer valores
        i = data_start
        while i < len(lines):
            line = lines[i].strip()
            if not line:
                i += 1
                continue
            parts = line.split()
            if len(parts) == 2:
                self.data[self.variables[0]].append(float(parts[1]))
                for v_idx in range(1, len(self.variables)):
                    val_line = lines[i + v_idx].strip()
                    self.data[self.variables[v_idx]].append(float(val_line))
                i += len(self.variables)
            else:
                i += 1
                
        # Convertir a arrays de numpy
        for var in self.variables:
            self.data[var] = np.array(self.data[var])
            
    def get_signal(self, name):
        name_lower = name.lower()
        for var in self.variables:
            if var.lower() == name_lower:
                return self.data[var]
        raise KeyError(f"Señal no encontrada: {name}. Disponibles: {self.variables}")

## 2. Compilación de Esquemáticos y Limpieza del Netlist
Compila los esquemáticos de Xschem para cargar los cambios más recientes del circuito de transistores y prepara el netlist limpio para la simulación.

In [3]:
import subprocess
import re
import os

print("1. Re-generando netlist de strongARM.sch...")
env = os.environ.copy()
env["PDK_ROOT"] = "/foss/pdks"
env["PDK"] = "gf180mcuD"
env["PDKPATH"] = "/foss/pdks/gf180mcuD"
env["PWD"] = "/foss/designs"
subprocess.run(
    ['/foss/tools/xschem/bin/xschem', '-n', '-q', '-o', '/foss/designs', '-p', '/foss/designs/strongARM.sch'],
    check=True, env=env
)

# Limpiar el netlist del comparador para remover comentarios de subcircuito
input_file = '/foss/designs/strongARM.spice'
output_file = '/foss/designs/strongARM_clean.spice'

with open(input_file, 'r') as f:
    content = f.read()

cleaned = content.replace('**.subckt', '.subckt')
cleaned = cleaned.replace('**.ends', '.ends')

with open(output_file, 'w') as f:
    f.write(cleaned)

print("Netlist limpio generado en strongARM_clean.spice")

1. Re-generando netlist de strongARM.sch...


Using run time directory XSCHEM_SHAREDIR = /foss/tools/xschem/share/xschem
Sourcing /foss/tools/xschem/share/xschem/xschemrc init file
Sourcing /headless/.xschem/xschemrc init file
pdk installation: using /foss/pdks
180MCU_MODELS: /foss/pdks/gf180mcuD/libs.tech/ngspice
setup_tcp_bespice: success : listening to TCP port: 2022



Netlist limpio generado en strongARM_clean.spice


## 3. Simulación de Esquinas PVT (45 Combinaciones)
Simula todas las combinaciones de:
- **Proceso (MOS):** `typical`, `ff`, `ss`, `fs`, `sf`
- **Voltaje (VDD):** `3.0V`, `3.3V`, `3.6V`
- **Temperatura:** `-40°C`, `27°C`, `125°C`

In [4]:
import itertools
import pandas as pd
import numpy as np
import subprocess
import re
import matplotlib.pyplot as plt

if SIMULATE_PVT:
    # Definición de las esquinas PVT
    mos_corners = ['typical', 'ff', 'ss', 'fs', 'sf']
    voltages = [3.0, 3.3, 3.6]
    temperatures = [-40, 27, 125]

    pvt_corners = []
    for mos, vdd, temp in itertools.product(mos_corners, voltages, temperatures):
        pvt_corners.append({
            'mos': mos,
            'vdd': vdd,
            'temp': temp
        })

    print(f"Iniciando simulación de esquinas PVT (Total: {len(pvt_corners)} esquinas)...\n")
    pvt_results = []

    for idx, corner in enumerate(pvt_corners):
        mos = corner['mos']
        vdd = corner['vdd']
        temp = corner['temp']

        # Generar netlist temporal para la esquina
        spice_pvt = f"""* Testbench PVT Corner Run {idx+1}
.include {PDK_PATH}/libs.tech/ngspice/design.ngspice
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice {mos}
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice res_typical
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice mimcap_typical
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice cap_mim

.include /foss/designs/strongARM_clean.spice

X1 CLK_NEG VDD GND ViN VIP Op On strongARM

V1 VDD GND {vdd}
V2 VIP GND {vdd/2 + 0.05}
V4 ViN GND {vdd/2 - 0.05}
V3 CLK_NEG GND pulse({vdd} 0 1n 100p 100p {CLK_PERIOD/2} {CLK_PERIOD})

.temp {temp}
.tran 5p {4 * CLK_PERIOD}

.measure tran tpd_ON
+ TRIG v(CLK_NEG) VAL={vdd/2} FALL=1 TD=500p
+ TARG v(On)      VAL={vdd/2} FALL=1

.measure tran I_avg AVG i(V1) FROM={1 * CLK_PERIOD} TO={4 * CLK_PERIOD}

.control
  run
  quit
.endc
.end
"""

        tb_path = f"/foss/designs/tb_pvt_delay_power.spice"
        with open(tb_path, 'w') as f:
            f.write(spice_pvt)

        res = subprocess.run(
            'source /headless/.bashrc && ngspice -b ' + tb_path,
            shell=True, executable='/bin/bash', capture_output=True, text=True
        )

        tpd = np.nan
        tpd_match = re.search(r'tpd_on\s*=\s*([0-9e.+-]+)', res.stdout, re.IGNORECASE)
        if tpd_match:
            tpd = float(tpd_match.group(1))

        i_avg = np.nan
        i_avg_match = re.search(r'i_avg\s*=\s*([0-9e.+-]+)', res.stdout, re.IGNORECASE)
        if i_avg_match:
            i_avg = float(i_avg_match.group(1))
        p_avg = abs(i_avg) * vdd if not np.isnan(i_avg) else np.nan

        # Simulación de Offset en la esquina PVT
        sim_time = 60 * CLK_PERIOD

        spice_pvt_offset = f"""* Testbench PVT Offset Run {idx+1}
.include {PDK_PATH}/libs.tech/ngspice/design.ngspice
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice {mos}
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice res_typical
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice mimcap_typical
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice cap_mim

.include /foss/designs/strongARM_clean.spice

X1 CLK_NEG VDD GND ViN VIP Op On strongARM

V1 VDD GND {vdd}
V2 VIP GND pwl(0 {vdd/2 - 0.08} {sim_time} {vdd/2 + 0.08})
V4 ViN GND {vdd/2}
V3 CLK_NEG GND pulse({vdd} 0 1n 100p 100p {CLK_PERIOD/2} {CLK_PERIOD})

.tran 50p {sim_time}
.measure tran VVIP_cross FIND v(VIP) WHEN v(On)={vdd/2} FALL=1

.control
  run
  quit
.endc
.end
"""
        tb_path_off = f"/foss/designs/tb_pvt_offset.spice"
        with open(tb_path_off, 'w') as f:
            f.write(spice_pvt_offset)

        res_off = subprocess.run(
            'source /headless/.bashrc && ngspice -b ' + tb_path_off,
            shell=True, executable='/bin/bash', capture_output=True, text=True
        )

        vvip_cross = np.nan
        cross_match = re.search(r'vvip_cross\s*=\s*([0-9e.+-]+)', res_off.stdout, re.IGNORECASE)
        if cross_match:
            vvip_cross = float(cross_match.group(1))

        vos = (vvip_cross - vdd/2) if not np.isnan(vvip_cross) else np.nan

        pvt_results.append({
            'mos': mos,
            'vdd': vdd,
            'temp': temp,
            'tpd (ps)': tpd * 1e12 if not np.isnan(tpd) else np.nan,
            'Pavg (uW)': p_avg * 1e6 if not np.isnan(p_avg) else np.nan,
            'Vos (mV)': vos * 1e3 if not np.isnan(vos) else np.nan
        })

        if (idx + 1) % 15 == 0 or (idx + 1) == len(pvt_corners):
            print(f"  -> Corrida {idx+1}/{len(pvt_corners)} completada.")

    df_pvt = pd.DataFrame(pvt_results)
    print("Simulación de esquinas PVT completada.")
else:
    df_pvt = pd.DataFrame()
    print("Simulación de esquinas PVT omitida por el usuario (SIMULATE_PVT = False).")

Iniciando simulación de esquinas PVT (Total: 45 esquinas)...

  -> Corrida 15/45 completada.
  -> Corrida 30/45 completada.
  -> Corrida 45/45 completada.
Simulación de esquinas PVT completada.


## 4. Simulación de Monte Carlo (200 Muestras - Mismatch Dinámico)
Simula 200 iteraciones con semillas aleatorias diferentes habilitando los parámetros estadísticos del PDK para simular variación de mismatch local de los transistores.

In [ ]:
import subprocess
import re
import pandas as pd
import numpy as np

mc_results = []
print(f"Iniciando simulación de Monte Carlo con {MC_SAMPLES} muestras...")

# Utilizaremos VDD nominal = 3.3V y Temp nominal = 27C
vdd = 3.3
temp = 27
vcm = 1.65

for run_idx in range(1, MC_SAMPLES + 1):
    # 1. Delay y Power
    spice_mc_dp = f"""* Testbench Monte Carlo DP Run {run_idx}
.include {PDK_PATH}/libs.tech/ngspice/design.ngspice
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice typical
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice res_typical
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice mimcap_typical
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice cap_mim

.include /foss/designs/strongARM_clean.spice

# Activar estadísticas
.param sw_stat_global=1
.param sw_stat_mismatch=1

X1 CLK_NEG VDD GND ViN VIP Op On strongARM

V1 VDD GND {vdd}
V2 VIP GND {vcm + 0.05}
V4 ViN GND {vcm - 0.05}
V3 CLK_NEG GND pulse({vdd} 0 1n 100p 100p {CLK_PERIOD/2} {CLK_PERIOD})

.temp {temp}
.tran 5p {4 * CLK_PERIOD}

.measure tran tpd_ON
+ TRIG v(CLK_NEG) VAL={vcm} FALL=1 TD=500p
+ TARG v(On)      VAL={vcm} FALL=1

.measure tran I_avg AVG i(V1) FROM={1 * CLK_PERIOD} TO={4 * CLK_PERIOD}

.control
  set rndseed = {run_idx}
  run
  quit
.endc
.end
"""
    tb_path = f"/foss/designs/tb_mc_dp.spice"
    with open(tb_path, 'w') as f:
        f.write(spice_mc_dp)

    res = subprocess.run(
        ['/foss/tools/ngspice/bin/ngspice', '-b', tb_path],
        capture_output=True, text=True
    )

    tpd = np.nan
    tpd_match = re.search(r'tpd_on\s*=\s*([0-9e.+-]+)', res.stdout, re.IGNORECASE)
    if tpd_match:
        tpd = float(tpd_match.group(1))

    i_avg = np.nan
    i_avg_match = re.search(r'i_avg\s*=\s*([0-9e.+-]+)', res.stdout, re.IGNORECASE)
    if i_avg_match:
        i_avg = float(i_avg_match.group(1))
    p_avg = abs(i_avg) * vdd if not np.isnan(i_avg) else np.nan

    # 2. Offset dinámico (Rampa que cubre ±80 mV en 60 ciclos para abarcar toda la distribución sin clipping)
    sim_time = 60 * CLK_PERIOD
    step_time = sim_time / 800
    
    spice_mc_off = f"""* Testbench Monte Carlo Offset Run {run_idx}
.include {PDK_PATH}/libs.tech/ngspice/design.ngspice
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice typical
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice res_typical
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice mimcap_typical
.lib {PDK_PATH}/libs.tech/ngspice/sm141064.ngspice cap_mim

.include /foss/designs/strongARM_clean.spice

# Activar estadísticas
.param sw_stat_global=1
.param sw_stat_mismatch=1

X1 CLK_NEG VDD GND ViN VIP Op On strongARM

V1 VDD GND {vdd}
V2 VIP GND pwl(0 {vcm - 0.08} {sim_time} {vcm + 0.08})
V4 ViN GND {vcm}
V3 CLK_NEG GND pulse({vdd} 0 1n 100p 100p {CLK_PERIOD/2} {CLK_PERIOD})

.temp {temp}
.tran 50p {sim_time}
.measure tran VVIP_cross FIND v(VIP) WHEN v(On)={vcm} FALL=1

.control
  set rndseed = {run_idx}
  run
  quit
.endc
.end
"""
    tb_path_off = f"/foss/designs/tb_mc_off.spice"
    with open(tb_path_off, 'w') as f:
        f.write(spice_mc_off)

    res_off = subprocess.run(
        ['/foss/tools/ngspice/bin/ngspice', '-b', tb_path_off],
        capture_output=True, text=True
    )
    
    vvip_cross = np.nan
    cross_match = re.search(r'vvip_cross\s*=\s*([0-9e.+-]+)', res_off.stdout, re.IGNORECASE)
    if cross_match:
        vvip_cross = float(cross_match.group(1))

    vos = (vvip_cross - vcm) if not np.isnan(vvip_cross) else np.nan

    mc_results.append({
        'Run': run_idx,
        'tpd (ps)': tpd * 1e12 if not np.isnan(tpd) else np.nan,
        'Pavg (uW)': p_avg * 1e6 if not np.isnan(p_avg) else np.nan,
        'Vos (mV)': vos * 1e3 if not np.isnan(vos) else np.nan
    })

    if run_idx % 20 == 0 or run_idx == MC_SAMPLES:
        print(f"  -> Corrida {run_idx}/{MC_SAMPLES} completada.")

df_mc = pd.DataFrame(mc_results)
print("Simulación de Monte Carlo completada.")

Iniciando simulación de Monte Carlo con 100 muestras...
  -> Corrida 20/100 completada.
  -> Corrida 40/100 completada.
  -> Corrida 60/100 completada.
  -> Corrida 80/100 completada.


## 5. Análisis Estadístico y Cálculo de Rendimiento (Yield)
Calcula la media ($\mu$), desviación estándar ($\sigma$) de los resultados de Monte Carlo, y calcula el Yield para cada especificación y para el sistema en conjunto.

In [ ]:
# Calcular rendimientos individuales
yield_delay = (df_mc['tpd (ps)'] <= SPEC_MAX_DELAY * 1e12).mean() * 100
yield_power = (df_mc['Pavg (uW)'] <= SPEC_MAX_POWER * 1e6).mean() * 100
yield_offset = (df_mc['Vos (mV)'].abs() <= SPEC_MAX_OFFSET * 1e3).mean() * 100

# Yield global (todas las specs juntas)
pass_delay = df_mc['tpd (ps)'] <= SPEC_MAX_DELAY * 1e12
pass_power = df_mc['Pavg (uW)'] <= SPEC_MAX_POWER * 1e6
pass_offset = df_mc['Vos (mV)'].abs() <= SPEC_MAX_OFFSET * 1e3
global_yield = (pass_delay & pass_power & pass_offset).mean() * 100

print("=== ESTADÍSTICAS Y RENDIMIENTO (YIELD) DE MONTE CARLO ===")
print("-" * 60)
print(f"Retardo (tpd):  Media = {df_mc['tpd (ps)'].mean():.2f} ps,  Std = {df_mc['tpd (ps)'].std():.2f} ps,  Yield = {yield_delay:.1f}%")
print(f"Potencia (Pavg): Media = {df_mc['Pavg (uW)'].mean():.2f} uW,  Std = {df_mc['Pavg (uW)'].std():.2f} uW,  Yield = {yield_power:.1f}%")
print(f"Offset (Vos):    Media = {df_mc['Vos (mV)'].mean():.2f} mV,  Std = {df_mc['Vos (mV)'].std():.2f} mV,  Yield = {yield_offset:.1f}%")
print("-" * 60)
print(f"Rendimiento Global del Comparador (Yield Total): {global_yield:.1f}%")

### 5.1 Tabla de Resumen de Resultados (Monte Carlo y Especificaciones)
Genera una tabla estilizada de resumen que consolida la media, desviación estándar, mejor caso, peor caso, límite de especificación y rendimiento (*Yield*) de cada métrica.

In [ ]:
# --- GENERAR TABLA BONITA DE MEJOR, TÍPICO Y PEOR CASO (MONTE CARLO)  y PVT ---
mc_summary_rows = [
    {
        'Métrica': 'Retardo (tpd) [ps]',
        'Media (Típico)': f"{df_mc['tpd (ps)'].mean():.2f}",
        'Desv. Est. (sigma)': f"{df_mc['tpd (ps)'].std():.2f}",
        'Mejor Caso (Mínimo)': f"{df_mc['tpd (ps)'].min():.2f}",
        'Peor Caso (Máximo)': f"{df_mc['tpd (ps)'].max():.2f}",
        'Especificación': f"< {SPEC_MAX_DELAY * 1e12:.0f} ps",
        'Yield': f"{yield_delay:.1f}%"
    },
    {
        'Métrica': 'Potencia (Pavg) [uW]',
        'Media (Típico)': f"{df_mc['Pavg (uW)'].mean():.2f}",
        'Desv. Est. (sigma)': f"{df_mc['Pavg (uW)'].std():.2f}",
        'Mejor Caso (Mínimo)': f"{df_mc['Pavg (uW)'].min():.2f}",
        'Peor Caso (Máximo)': f"{df_mc['Pavg (uW)'].max():.2f}",
        'Especificación': f"< {SPEC_MAX_POWER * 1e6:.0f} uW",
        'Yield': f"{yield_power:.1f}%"
    },
    {
        'Métrica': 'Offset (Vos) [mV]',
        'Media (Típico)': f"{df_mc['Vos (mV)'].mean():.2f}",
        'Desv. Est. (sigma)': f"{df_mc['Vos (mV)'].std():.2f}",
        'Mejor Caso (Mínimo Abs)': f"{df_mc['Vos (mV)'].abs().min():.2f}",
        'Peor Caso (Máximo Abs)': f"{df_mc['Vos (mV)'].abs().max():.2f}",
        'Especificación': f"± {SPEC_MAX_OFFSET * 1e3:.0f} mV",
        'Yield': f"{yield_offset:.1f}%"
    },
]
df_mc_summary = pd.DataFrame(mc_summary_rows)
from IPython.display import display
display(df_mc_summary.style.set_properties(**{'text-align': 'center'}).hide(axis='index'))

## 6. Visualización de Histogramas
Genera gráficos de distribución con límites de especificación indicados.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# 1. Histograma de Delay
ax1.hist(df_mc['tpd (ps)'].dropna(), bins=20, color='skyblue', edgecolor='black', alpha=0.8)
ax1.axvline(SPEC_MAX_DELAY * 1e12, color='red', linestyle='--', linewidth=2, label=f'Spec Max: {SPEC_MAX_DELAY*1e12:.0f} ps')
ax1.set_xlabel('tpd (ps)')
ax1.set_ylabel('Frecuencia')
ax1.set_title('Distribución de Retardo de Propagación')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Histograma de Potencia
ax2.hist(df_mc['Pavg (uW)'].dropna(), bins=20, color='lightgreen', edgecolor='black', alpha=0.8)
ax2.axvline(SPEC_MAX_POWER * 1e6, color='red', linestyle='--', linewidth=2, label=f'Spec Max: {SPEC_MAX_POWER*1e6:.0f} uW')
ax2.set_xlabel('Pavg (uW)')
ax2.set_title('Distribución de Potencia Promedio')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Histograma de Offset
ax3.hist(df_mc['Vos (mV)'].dropna(), bins=20, color='salmon', edgecolor='black', alpha=0.8)
ax3.axvline(SPEC_MAX_OFFSET * 1e3, color='red', linestyle='--', linewidth=2, label=f'Spec Max: +{SPEC_MAX_OFFSET*1e3:.0f} mV')
ax3.axvline(-SPEC_MAX_OFFSET * 1e3, color='red', linestyle='--', linewidth=2, label=f'Spec Min: -{SPEC_MAX_OFFSET*1e3:.0f} mV')
ax3.set_xlabel('Vos (mV)')
ax3.set_title('Distribución de Offset Dinámico')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparador_mc_plots.png', dpi=150)
plt.show()

### 6.1 Histograma de Offset Detallado (Estilo Cadence Virtuoso)
Genera una gráfica detallada enfocada únicamente en la distribución de Offset Dinámico ($V_{OS}$). 
Incluye el histograma de frecuencias en la parte superior y la proyección de cada muestra individual como un punto (*scatter plot* con jitter vertical) en la parte inferior, emulando la visualización clásica de simulaciones de Monte Carlo en **Cadence Virtuoso (ADEXL / Maestro)**. La imagen se guarda automáticamente como `comparador_offset_montecarlo.png`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extraer datos de offset
vos_data = df_mc['Vos (mV)'].dropna()
mu = vos_data.mean()
sigma = vos_data.std()
yield_val = (vos_data.abs() <= SPEC_MAX_OFFSET * 1e3).mean() * 100

# Crear la figura con dos subplots acoplados verticalmente (estilo Virtuoso ADEXL)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, 
                               gridspec_kw={'height_ratios': [4, 1]})
fig.subplots_adjust(hspace=0.05) # Sin espacio vertical entre gráficos

# 1. Histograma superior
n, bins, patches = ax1.hist(vos_data, bins=25, color='salmon', edgecolor='black', alpha=0.8, label='Distribución')
ax1.axvline(SPEC_MAX_OFFSET * 1e3, color='red', linestyle='--', linewidth=2, 
            label=f'Spec Max: +{SPEC_MAX_OFFSET*1e3:.1f} mV')
ax1.axvline(-SPEC_MAX_OFFSET * 1e3, color='red', linestyle='--', linewidth=2, 
            label=f'Spec Min: -{SPEC_MAX_OFFSET*1e3:.1f} mV')
ax1.axvline(mu, color='blue', linestyle='-.', linewidth=1.5, label=f'Media (mu): {mu:.2f} mV')

# Añadir caja de texto con estadísticas básicas
stats_text = (
    f"Muestras: {len(vos_data)}\n"
    f"Media ($\mu$): {mu:.2f} mV\n"
    f"Desv. Est. ($\sigma$): {sigma:.2f} mV\n"
    f"Yield: {yield_val:.1f}%"
)
ax1.text(0.05, 0.95, stats_text, transform=ax1.transAxes, fontsize=10,
         verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.8, ec='gray'))

ax1.set_title('Histograma de Offset Dinámico - Simulación de Monte Carlo', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frecuencia (Conteos)')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# 2. Puntos de muestras individuales en el panel inferior (Estilo Cadence Virtuoso)
# Se añade un pequeño jitter aleatorio en el eje Y para evitar solapamiento de puntos y ver la densidad
np.random.seed(42) # Semilla fija para consistencia gráfica
jitter = np.random.uniform(-0.15, 0.15, size=len(vos_data))
ax2.scatter(vos_data, jitter, color='darkred', marker='o', s=12, alpha=0.5, label='Muestras individuales')
ax2.axvline(SPEC_MAX_OFFSET * 1e3, color='red', linestyle='--', linewidth=1.5, alpha=0.5)
ax2.axvline(-SPEC_MAX_OFFSET * 1e3, color='red', linestyle='--', linewidth=1.5, alpha=0.5)
ax2.axvline(mu, color='blue', linestyle='-.', linewidth=1, alpha=0.5)

# Formatear el panel inferior
ax2.set_ylim(-0.5, 0.5)
ax2.get_yaxis().set_visible(False) # Ocultar el eje Y
ax2.spines['top'].set_visible(False)
ax2.spines['left'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_xlabel('Tensión de Offset Dinámico, $V_{OS}$ (mV)', fontsize=10)
ax2.grid(True, axis='x', alpha=0.3)

# Guardar la gráfica como imagen independiente en el directorio del comparador
plt.savefig('comparador_offset_montecarlo.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Presentación de Resultados Estilo Virtuoso ADEXL
Muestra una tabla estilizada de resultados PVT indicando si pasa (PASS) o falla (FAIL) cada combinación.

In [ ]:
from IPython.display import display

def style_pvt_table(row):
    styles = [''] * len(row)
    idx_tpd = row.index.get_loc('tpd (ps)')
    idx_pavg = row.index.get_loc('Pavg (uW)')
    idx_vos = row.index.get_loc('Vos (mV)')

    # Evaluar Delay
    tpd_val = row['tpd (ps)']
    if pd.isna(tpd_val) or tpd_val > SPEC_MAX_DELAY * 1e12:
        styles[idx_tpd] = 'background-color: #f8d7da; color: #721c24; font-weight: bold;'
    else:
        styles[idx_tpd] = 'background-color: #d4edda; color: #155724;'

    # Evaluar Potencia
    pavg_val = row['Pavg (uW)']
    if pd.isna(pavg_val) or pavg_val > SPEC_MAX_POWER * 1e6:
        styles[idx_pavg] = 'background-color: #f8d7da; color: #721c24; font-weight: bold;'
    else:
        styles[idx_pavg] = 'background-color: #d4edda; color: #155724;'

    # Evaluar Offset
    vos_val = row['Vos (mV)']
    if pd.isna(vos_val) or abs(vos_val) > SPEC_MAX_OFFSET * 1e3:
        styles[idx_vos] = 'background-color: #f8d7da; color: #721c24; font-weight: bold;'
    else:
        styles[idx_vos] = 'background-color: #d4edda; color: #155724;'

    return styles

if 'df_pvt' in locals() and not df_pvt.empty:
    print("=== REPORTE DE ESQUINAS PVT (ADEXL STYLE) ===")
    styled_pvt = df_pvt.style.apply(style_pvt_table, axis=1)
    display(styled_pvt)
else:
    print("=== REPORTE DE ESQUINAS PVT OMITIDO ===")
    print("La simulación de esquinas PVT no fue ejecutada (SIMULATE_PVT = False).")

# =====================================================================
# RESUMEN DE MEJORES Y PEORES CASOS (PVT)
# =====================================================================
print("\n=== RESUMEN DE MEJORES Y PEORES CASOS (PVT) ===")
print("-" * 60)

# 1. Delay (tpd)
df_tpd = df_pvt.dropna(subset=['tpd (ps)'])
if not df_tpd.empty:
    best_tpd = df_tpd.loc[df_tpd['tpd (ps)'].idxmin()]
    worst_tpd = df_tpd.loc[df_tpd['tpd (ps)'].idxmax()]
    best_tpd_corner = f"{best_tpd['mos'].upper()}_{best_tpd['vdd']}V_{best_tpd['temp']}C"
    worst_tpd_corner = f"{worst_tpd['mos'].upper()}_{worst_tpd['vdd']}V_{worst_tpd['temp']}C"
    print("Retardo (tpd):")
    print(f"  * MEJOR caso: {best_tpd['tpd (ps)']:.2f} ps en esquina {best_tpd_corner}")
    print(f"  * PEOR caso:  {worst_tpd['tpd (ps)']:.2f} ps en esquina {worst_tpd_corner}")
else:
    print("Retardo (tpd): No hay datos disponibles.")
print("-" * 60)

# 2. Potencia (Pavg)
df_pavg = df_pvt.dropna(subset=['Pavg (uW)'])
if not df_pavg.empty:
    best_pavg = df_pavg.loc[df_pavg['Pavg (uW)'].idxmin()]
    worst_pavg = df_pavg.loc[df_pavg['Pavg (uW)'].idxmax()]
    best_pavg_corner = f"{best_pavg['mos'].upper()}_{best_pavg['vdd']}V_{best_pavg['temp']}C"
    worst_pavg_corner = f"{worst_pavg['mos'].upper()}_{worst_pavg['vdd']}V_{worst_pavg['temp']}C"
    print("Potencia (Pavg):")
    print(f"  * MEJOR caso (Menor consumo): {best_pavg['Pavg (uW)']:.2f} uW en esquina {best_pavg_corner}")
    print(f"  * PEOR caso (Mayor consumo):  {worst_pavg['Pavg (uW)']:.2f} uW en esquina {worst_pavg_corner}")
else:
    print("Potencia (Pavg): No hay datos disponibles.")
print("-" * 60)

# 3. Offset (Vos)
df_vos = df_pvt.dropna(subset=['Vos (mV)'])
if not df_vos.empty:
    abs_vos = df_vos['Vos (mV)'].abs()
    best_vos = df_vos.loc[abs_vos.idxmin()]
    worst_vos = df_vos.loc[abs_vos.idxmax()]
    best_vos_corner = f"{best_vos['mos'].upper()}_{best_vos['vdd']}V_{best_vos['temp']}C"
    worst_vos_corner = f"{worst_vos['mos'].upper()}_{worst_vos['vdd']}V_{worst_vos['temp']}C"
    print("Offset (Vos):")
    print(f"  * MEJOR caso (Menor magnitud): {best_vos['Vos (mV)']:.2f} mV en esquina {best_vos_corner}")
    print(f"  * PEOR caso (Mayor magnitud):  {worst_vos['Vos (mV)']:.2f} mV en esquina {worst_vos_corner}")
else:
    print("Offset (Vos): No hay datos disponibles.")
print("-" * 60)